# Stage 1: drop empty / constant / redundant columns (bronze -> silver)

The **only** transformation in this notebook is dropping columns. No rows are removed, no values are changed (dates stay Excel serials, thresholds stay as they are).

| Column dropped | Reason |
|---|---|
| `ID_GXP_APOPO` | 100% empty |
| `ReadTotalSnifftime` | constant (always True) |
| `Trainer`, `Documenter`, `Handler` | only 30 populated rows |
| `ID_CONFIGURATION` | one-to-one with `CONFIGURATION_NAME` |
| `LEVEL_NAME` | text label fully determined by `ID_BL_DOTS` + `SAMPLE_TYPE` (the surrogate marker SUG/SAL/BTE/CAM is also in `SAMPLE_TYPE`) |

Input: `../data/bronze/data_bronze.xlsb` | Output: `../data/silver/data_stage1.parquet`

In [ ]:
from pathlib import Path

import pandas as pd

BRONZE = Path("../data/bronze/data_bronze.xlsb")
SILVER = Path("../data/silver/data_stage1.parquet")

DROP_COLUMNS = {
    "ID_GXP_APOPO": "100% empty",
    "ReadTotalSnifftime": "constant (always True)",
    "Trainer": "only 30 populated rows",
    "Documenter": "only 30 populated rows",
    "Handler": "only 30 populated rows",
    "ID_CONFIGURATION": "one-to-one with CONFIGURATION_NAME",
    "LEVEL_NAME": "text version of ID_BL_DOTS + SAMPLE_TYPE",
}

## 1. Load bronze

In [ ]:
bronze = pd.read_excel(BRONZE, engine="pyxlsb")
print(bronze.shape)

## 2. Check the reasons before dropping

In [ ]:
checks = pd.DataFrame({
    "column": list(DROP_COLUMNS),
    "reason": list(DROP_COLUMNS.values()),
    "n_populated": [bronze[c].notna().sum() for c in DROP_COLUMNS],
    "n_unique": [bronze[c].nunique() for c in DROP_COLUMNS],
})
checks

In [ ]:
# ID_CONFIGURATION must map one-to-one to CONFIGURATION_NAME; LEVEL_NAME is fully determined by ID_BL_DOTS + SAMPLE_TYPE
assert bronze.groupby("ID_CONFIGURATION")["CONFIGURATION_NAME"].nunique().max() == 1
assert bronze.groupby("CONFIGURATION_NAME")["ID_CONFIGURATION"].nunique().max() == 1
assert bronze.groupby(["ID_BL_DOTS", "SAMPLE_TYPE"], dropna=False)["LEVEL_NAME"].nunique().max() == 1
assert bronze["ID_GXP_APOPO"].isna().all()
assert bronze["ReadTotalSnifftime"].nunique() == 1
print("all drop reasons confirmed")

## 3. Drop columns and write silver

In [ ]:
silver = bronze.drop(columns=list(DROP_COLUMNS))

assert len(silver) == len(bronze)  # no rows removed
print(f"columns: {bronze.shape[1]} -> {silver.shape[1]} | rows: {len(silver):,}")

SILVER.parent.mkdir(parents=True, exist_ok=True)
silver.to_parquet(SILVER, index=False)
print("written:", SILVER.resolve())

In [ ]:
silver.dtypes.to_frame("dtype")